In [2]:
import numpy as np
import pandas as pd
import sklearn

from sklearn.linear_model import LinearRegression, LogisticRegression, Lasso, Ridge

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.metrics import root_mean_squared_error, mean_absolute_error 
from sklearn.metrics import accuracy_score, recall_score, confusion_matrix, precision_score, roc_auc_score


In [1]:

df = pd.read_csv("data/geo_all_channels.csv")
df.head(6)

,Unnamed: 0,geo,time,Channel0_impression,Channel1_impression,Channel2_impression,Channel3_impression,Channel4_impression,competitor_sales_control,sentiment_score_control,Channel0_spend,Channel1_spend,Channel2_spend,Channel3_spend,Channel4_spend,Organic_channel0_impression,Promo,conversions,revenue_per_conversion,population
0,0,Geo0,2021-01-25,280668,0,0,470611,108010,-1.338765,0.115581,2058.0608,0.00000,0.00000,3667.3965,841.60440,97320,0.000000,1954576.8,0.020055,136670.94
1,1,Geo0,2021-02-01,366206,182108,19825,527702,252506,0.893645,0.944224,2685.2874,1755.74540,147.31808,4112.2974,1967.50440,201441,0.000000,2064249.6,0.020103,136670.94
2,2,Geo0,2021-02-08,197565,230170,0,393618,184061,-0.284549,-1.290579,1448.6895,2219.12230,0.00000,3067.4023,1434.18700,0,0.683819,2086382.8,0.019929,136670.94
3,3,Geo0,2021-02-15,140990,66643,0,326034,201729,-1.034740,-1.084514,1033.8406,642.52057,0.00000,2540.7310,1571.85450,0,1.289055,2826431.5,0.019987,136670.94
4,4,Geo0,2021-02-22,399116,164991,0,381982,153973,-0.319276,-0.017503,2926.6072,1590.71640,0.00000,2976.7249,1199.74400,0,0.227739,3551929.2,0.020000,136670.94
5,5,Geo0,2021-03-01,219462,149254,0,417941,41573,-0.652695,-0.302114,1609.2542,1438.99240,0.00000,3256.9475,323.93317,0,0.000000,2241229.2,0.020142,136670.94


In [3]:
print(df.isna().any(axis = None))

False


In [ ]:
df.columns


In [7]:
df['revenue'] = df['conversions'] * df['revenue_per_conversion']

In [8]:
df["revenue"].describe()

count      6240.000000
mean     211330.038099
std      121605.908396
min        9742.282233
25%      114323.174381
50%      193995.582649
75%      287918.213647
max      739823.338940
Name: revenue, dtype: float64

In [10]:
print(df["time"].dtype)

str


In [11]:
df["time"] = pd.to_datetime(df["time"])

In [ ]:
summary = (
    df.groupby("geo")
      .agg(
          n_obs=("time", "count"),
          time_min=("time", "min"),
          time_max=("time", "max"),
          time_range=("time", lambda x: x.max() - x.min()),
          # time_var=("time", "var")
      )
)

print(summary)

In [ ]:
sorted(df["time"].unique())

In [16]:
times = df["time"].sort_values().unique()
diffs = pd.Series(times).diff()

diffs.value_counts()

7 days    155
Name: count, dtype: int64

In [17]:
df

,Unnamed: 0,geo,time,Channel0_impression,Channel1_impression,Channel2_impression,Channel3_impression,Channel4_impression,competitor_sales_control,sentiment_score_control,...,Channel1_spend,Channel2_spend,Channel3_spend,Channel4_spend,Organic_channel0_impression,Promo,conversions,revenue_per_conversion,population,revenue
0,0,Geo0,2021-01-25,280668,0,0,470611,108010,-1.338765,0.115581,...,0.00000,0.00000,3667.3965,841.6044,97320,0.000000,1954576.8,0.020055,136670.94,39198.556898
1,1,Geo0,2021-02-01,366206,182108,19825,527702,252506,0.893645,0.944224,...,1755.74540,147.31808,4112.2974,1967.5044,201441,0.000000,2064249.6,0.020103,136670.94,41497.960631
2,2,Geo0,2021-02-08,197565,230170,0,393618,184061,-0.284549,-1.290579,...,2219.12230,0.00000,3067.4023,1434.1870,0,0.683819,2086382.8,0.019929,136670.94,41579.088854
3,3,Geo0,2021-02-15,140990,66643,0,326034,201729,-1.034740,-1.084514,...,642.52057,0.00000,2540.7310,1571.8545,0,1.289055,2826431.5,0.019987,136670.94,56492.861509
4,4,Geo0,2021-02-22,399116,164991,0,381982,153973,-0.319276,-0.017503,...,1590.71640,0.00000,2976.7249,1199.7440,0,0.227739,3551929.2,0.020000,136670.94,71039.827175
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6235,6235,Geo9,2023-12-18,1464367,857806,0,1956612,0,-0.719707,0.331813,...,8270.30700,0.00000,15247.5650,0.0000,541384,0.000000,15234047.0,0.020122,513695.22,306538.031265
6236,6236,Geo9,2023-12-25,953124,489442,0,1427191,0,-1.053569,0.519312,...,4718.82370,0.00000,11121.8720,0.0000,0,0.000000,18156892.0,0.020086,513695.22,364698.043573
6237,6237,Geo9,2024-01-01,0,0,213133,2846124,1310260,0.548388,-0.569818,...,0.00000,1583.77530,22179.3900,10209.4300,0,0.388046,10182924.0,0.019855,513695.22,202180.082362
6238,6238,Geo9,2024-01-08,1209242,497181,0,1287116,967957,-0.385787,0.364464,...,4793.43700,0.00000,10030.2900,7542.2354,709495,0.258418,12583538.0,0.020161,513695.22,253702.724549
